In [ ]:
from pathlib import Path
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CODEBASE_DIR = Path('/content/drive/MyDrive/predicting-the-future-effect-of-optimizer-memory')  
assert (CODEBASE_DIR / 'requirements.txt').is_file(), 'Upload and unzip the code folder to Drive first.'
os.chdir(CODEBASE_DIR)
sys.path.insert(0, str(CODEBASE_DIR))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])


In [ ]:
RUN_MODE = 'smoke'   
FORCE = False        
RUN_PREFLIGHT_AUDIT = True
RUN_DATASET_PROBE = True
RUN_FIGURES = True
RUN_FROZEN_BASELINE = True
PAPER_MATERIAL_EXPORT_DIR = ''  


DATASETS = ('split_cifar10',)          
ARCHITECTURES = ('smallcnn', 'resnet18')
OPTIMIZERS = ('heavy_ball', 'adam')
SEEDS = (0, 1, 2, 3, 4)

ROOT_NAME = 'results/rttp_full_grid'
RUN_NAME = 'rttp_full_grid'

print('RUN_MODE:', RUN_MODE)

In [ ]:
from rttp_grid.config import GridConfig

if RUN_MODE == 'smoke':
    cfg = GridConfig(
        run_name='rttp_full_grid_smoke', root_name='rttp_full_grid_smoke',
        datasets=('split_cifar10',), architectures=('smallcnn',), optimizers=('heavy_ball', 'adam'), seeds=(0,),
        cifar_train_per_class=50, cifar_test_per_class=50, batch_size=64,
        boundary_epochs_smallcnn_cifar=1, horizons_sgd=(1, 3), horizons_adam=(1, 3),
        sgd_response_lrs_cifar=(0.01,), sgd_momenta=(0.9,),
        adam_response_lrs_cifar=(3e-5,), adam_memory_components=('m', 'rms_global'),
        n_directions=1, num_workers=0, force=FORCE,
        run_preflight_audit=RUN_PREFLIGHT_AUDIT,
        run_dataset_probe=RUN_DATASET_PROBE,
        run_figures=False,
        run_frozen_baseline=RUN_FROZEN_BASELINE,
        paper_material_dir=PAPER_MATERIAL_EXPORT_DIR,
    )
elif RUN_MODE == 'full':
    cfg = GridConfig(
        run_name=RUN_NAME, root_name=ROOT_NAME, force=FORCE,
        run_preflight_audit=RUN_PREFLIGHT_AUDIT,
        run_dataset_probe=RUN_DATASET_PROBE,
        run_figures=RUN_FIGURES,
        run_frozen_baseline=RUN_FROZEN_BASELINE,
        paper_material_dir=PAPER_MATERIAL_EXPORT_DIR,
    )
elif RUN_MODE == 'custom':
    cfg = GridConfig(
        run_name=RUN_NAME, root_name=ROOT_NAME,
        datasets=DATASETS, architectures=ARCHITECTURES, optimizers=OPTIMIZERS, seeds=SEEDS,
        force=FORCE,
        run_preflight_audit=RUN_PREFLIGHT_AUDIT,
        run_dataset_probe=RUN_DATASET_PROBE,
        run_figures=RUN_FIGURES,
        run_frozen_baseline=RUN_FROZEN_BASELINE,
        paper_material_dir=PAPER_MATERIAL_EXPORT_DIR,
    )
else:
    raise ValueError(RUN_MODE)

config_path = Path('active_colab_config.json')
cfg.to_json(config_path)
print(config_path.read_text())

In [ ]:
from rttp_grid.runner import run_grid

output_root = run_grid(cfg)
print('output_root:', output_root)

In [ ]:
from rttp_grid.runner import summarize_outputs, make_context






In [ ]:
import pandas as pd
from pathlib import Path

from rttp_grid.utils import get_root
root = get_root(cfg)
for name in ['quality_report.csv', 'paper_table_full_grid.csv', 'frozen_best_comparison.csv', 'frozen_comparison_by_scale.csv', 'boundary_summary.csv']:
    path = root / 'tables' / name
    print('\n', path)
    if path.exists():
        display(pd.read_csv(path).head(20))
    else:
        print('missing')

print('archives:')
for p in sorted((root / 'archives').glob('*.zip'))[-5:]:
    print(p)

for subdir, name in [('audits', 'preflight_audit_summary.csv'), ('diagnostics', 'dataset_probe.csv')]:
    path = root / subdir / name
    print('\n', path)
    if path.exists():
        display(pd.read_csv(path).head(20))
    else:
        print('missing')